In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
print("libraries ready!")

libraries ready!


In [2]:
df = pd.read_csv('../data/processed/weather_clean.csv')
print(df.shape)
print(df['will_rain'].value_counts())

C:\Users\SAMPATH\AppData\Local\Temp\ipykernel_32536\792926546.py:1: DtypeWarning: Columns (0: station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/weather_clean.csv')


(27621770, 13)
will_rain
0    20009891
1     7611879
Name: count, dtype: int64


In [3]:
features = ['avg_temp_c', 'min_temp_c', 'max_temp_c', 'avg_wind_speed_kmh', 'month', 'day', 'year']
x = df[features]
y = df['will_rain']

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: (27621770, 7)
y shape: (27621770,)


In [4]:
x_sample = x.sample(n=500000, random_state=42)
y_sample = y[x_sample.index]
x_train, x_test, y_train, y_test = train_test_split(
    x_sample, y_sample, test_size=0.2, random_state=42 
)
print("train:", x_train.shape)
print("test:", x_test.shape)

train: (400000, 7)
test: (100000, 7)


In [5]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(x_train, y_train)
print("xgboost training done!")

xgboost training done!


In [6]:
y_pred = xgb_model.predict(x_test)
print(classification_report(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.94      0.84     72372
           1       0.60      0.23      0.33     27628

    accuracy                           0.74    100000
   macro avg       0.68      0.59      0.59    100000
weighted avg       0.72      0.74      0.70    100000

AUC-ROC: 0.5858740526231319


In [7]:
xgb_model2 = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=3
)

xgb_model2.fit(x_train, y_train)
y_pred2 = xgb_model2.predict(x_test)
print(classification_report(y_test, y_pred2))
print("AUC-ROC:", roc_auc_score(y_test, y_pred2))

              precision    recall  f1-score   support

           0       0.87      0.56      0.68     72372
           1       0.41      0.79      0.54     27628

    accuracy                           0.62    100000
   macro avg       0.64      0.67      0.61    100000
weighted avg       0.74      0.62      0.64    100000

AUC-ROC: 0.6736922884979093


In [8]:
os.makedirs('../models/xgboost', exist_ok=True)
joblib.dump(xgb_model2, '../models/xgboost/model.pkl')
joblib.dump(features, '../models/xgboost/features.pkl')
print("xgboost saved!")

xgboost saved!
